# SpaceX Falcon 9 — Data Wrangling

## Project objective
Prepare launch-level data for exploratory analysis and supervised learning by checking data quality, examining launch and landing categories, and creating a clearly defined binary target.

**Target definition:** `Class = 1` when the first-stage landing outcome is explicitly successful; `Class = 0` when the outcome is explicitly unsuccessful. Unknown or unattempted landing outcomes are not silently treated as successful.

**Tools:** Python and Pandas.

In [ ]:
import pandas as pd
import numpy as np

DATA_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv"
df = pd.read_csv(DATA_URL)

print(f"Dataset shape: {df.shape}")
display(df.head(10))
print("\nColumn names:")
print(df.columns.tolist())

## 1. Inspect missing values

Calculate both the number and percentage of missing values in each column. This helps determine whether a field needs cleaning or should be excluded from a particular analysis.

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)

display(missing_summary)

## 2. Inspect data types and duplicates

Review numeric and categorical columns, then check whether duplicate launch records are present. Do not drop duplicates automatically without confirming what defines a unique launch.

In [ ]:
display(df.dtypes.to_frame("data_type"))
duplicate_count = df.duplicated().sum()
print(f"Exact duplicate rows: {duplicate_count:,}")

## 3. Explore launch sites, orbits, and landing outcomes

Frequency tables provide a baseline view of the dataset before feature engineering.

In [ ]:
for column in ["LaunchSite", "Orbit", "Outcome"]:
    print(f"\n{column} frequency:")
    display(df[column].value_counts(dropna=False).rename("launch_count").to_frame())

## 4. Define the landing-success target

The source column `Outcome` contains a mixture of successful, unsuccessful, and unknown/unattempted outcomes. Define the positive and negative classes explicitly, and flag records whose outcome cannot be classified from the label.

This avoids relying on category positions in a sorted frequency table, which can change when the dataset changes.

In [ ]:
successful_outcomes = {"True ASDS", "True Ocean", "True RTLS"}
unsuccessful_outcomes = {"False ASDS", "False Ocean", "False RTLS"}

df["Class"] = np.select(
    [
        df["Outcome"].isin(successful_outcomes),
        df["Outcome"].isin(unsuccessful_outcomes)
    ],
    [1, 0],
    default=np.nan
)

df["Class"] = df["Class"].astype("Int64")

print("Target label counts (including unknown outcomes):")
display(df["Class"].value_counts(dropna=False).rename("launch_count").to_frame())

classified = df["Class"].dropna()
if len(classified):
    print(f"Success rate among explicitly classified outcomes: {classified.mean():.1%}")
else:
    print("No explicitly classified landing outcomes were found.")

unknown_outcomes = sorted(df.loc[df["Class"].isna(), "Outcome"].dropna().unique().tolist())
print("Unclassified outcome labels:", unknown_outcomes)

## 5. Review launch counts by site and orbit

These summaries help identify which groups dominate the dataset. The count of records in a group should be considered when interpreting later success rates.

In [ ]:
launches_by_site = df["LaunchSite"].value_counts().rename_axis("LaunchSite").reset_index(name="launch_count")
launches_by_orbit = df["Orbit"].value_counts().rename_axis("Orbit").reset_index(name="launch_count")

display(launches_by_site)
display(launches_by_orbit)

## 6. Create an analysis-ready dataset

Keep the original data available for auditability. Create a separate dataset containing only records with known target labels for supervised modeling. Missing predictor values are reported rather than filled with an arbitrary value; imputation should be performed inside a model pipeline after the train/test split.

In [ ]:
model_df = df.loc[df["Class"].notna()].copy()
model_df["Class"] = model_df["Class"].astype("int64")

print("Original shape:", df.shape)
print("Labeled modeling shape:", model_df.shape)
print("\nMissing values in labeled data:")
display(model_df.isna().sum().sort_values(ascending=False).to_frame("missing_values"))

## 7. Validate the target and export

Check that the target contains only 0 and 1, then export the labeled data for the next stage.

In [ ]:
assert set(model_df["Class"].unique()).issubset({0, 1})
assert model_df["Class"].notna().all()

model_df.to_csv("dataset_part_2.csv", index=False)
print("Saved dataset_part_2.csv")
print("Class distribution:")
display(model_df["Class"].value_counts(normalize=True).rename("proportion").to_frame())

## Data-wrangling decisions and limitations

- Landing outcomes are mapped by explicit labels, not by the position of a value in a frequency table.
- Unknown or unattempted outcomes are excluded from the labeled modeling export and counted separately.
- Missing predictor values are not automatically imputed here. For predictive modeling, fit imputation and scaling steps using training data only to avoid data leakage.
- The target is first-stage landing success, not overall mission success.

## Reproducibility
- Source: IBM Skills Network SpaceX dataset (loaded in the first code cell).
- Output: `dataset_part_2.csv`, containing records with a known binary landing label.
- Run all cells from top to bottom to regenerate the output.